# Tables S10 and S11: MagNET-Zero/MagNET-PCM scaling factors

Per-solvent linear coefficients (intercept, stationary, pcm) mapping MagNET-Zero shieldings plus a
MagNET-PCM correction to predicted shifts: ¹H (S10) and ¹³C (S11).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "data/scaling_factors", "data/applications",
           "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import numpy as np
import pandas as pd
import scaling_factors
import scaling_factors_reader
import build_composite_model
import paths
from applications_reader import Applications

In [ ]:
DELTA22 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def document_path(name):
    os.makedirs("documents", exist_ok=True)
    return os.path.join("documents", name)

In [ ]:
tables = scaling_factors.published_scaling_tables()   # {"H": Table S10, "C": Table S11}
print("Table S10 (1H):"); display(tables["H"])
print("Table S11 (13C):"); display(tables["C"])

out = document_path("si_table_s10_s11_scaling.xlsx")
with pd.ExcelWriter(out) as writer:
    tables["H"].reset_index().to_excel(writer, sheet_name="Table S10 (1H)", index=False)
    tables["C"].reset_index().to_excel(writer, sheet_name="Table S11 (13C)", index=False)
print("wrote", os.path.relpath(out, REPO))

## Reproducibility check: re-derive from delta-22

In [ ]:
derived = scaling_factors.build_scaling_tables(DELTA22, XLSX)
for nucleus in ("H", "C"):
    p = tables[nucleus]
    d = derived[nucleus].reindex(p.index)[p.columns]
    max_dev = float(np.abs(p.values - d.values).max())
    print(f"{nucleus}: largest published-vs-rederived deviation = {max_dev:.2e}")
    assert max_dev < 1e-5, f"{nucleus} scaling table diverged from the published values"
print("both tables reproduce from delta-22")

## Deployment tables with reflection symmetrization

The tables above match the published SI exactly. For serving new molecules, the recommended tables
average each prediction with its mirror image, correcting a reflection-parity error. Reproducing
them needs the model checkpoints; the shipped tables are shown below.

In [ ]:
symmetrized = scaling_factors_reader.load_symmetrized_tables()
print("Symmetrized deployment Table S10 (1H):"); display(symmetrized["H"])
print("Symmetrized deployment Table S11 (13C):"); display(symmetrized["C"])

## Composite-model coefficients (Figure 5C/5D, SI S15)

The natural-products figures use a larger family of per-solvent coefficients fit on delta-22 (the
`composite_model` group in `applications.hdf5`): OLS fits, 1000-seed bootstrap resamples, their RMSE
distributions, and the PCM conversion factors. The group regenerates from released inputs alone.

In [ ]:
reader = Applications(paths.dataset_file("applications", root=REPO))
regenerated = build_composite_model.regenerate(reader)
max_dev = build_composite_model.verify(reader, regenerated)
print(f"composite_model largest regenerated-vs-stored deviation = {max_dev:.2e}")